<img src="../static/agent-overview.svg" alt="Agent-Overview" width="100%" height="800px">

## Base of project

In [1]:
if input() == 'y':
    !python -m pip install --upgrade pip

    %pip uninstall ragas

    %pip freeze > requirements.txt

In [2]:
from typing import TypedDict, Annotated, Literal
from datetime import datetime as dt
from dotenv import load_dotenv
import pandas as pd
import os, uuid

from pydantic import BaseModel, Field

from langchain_core.vectorstores import InMemoryVectorStore as imVectors
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from langchain_core.documents import Document
from langchain_core.tools import tool

from videoIngestion import VideoIngestionPipeline

In [3]:
load_dotenv()
GEMINI_API = os.getenv("PAID_GEMINI_API") # PAID_GEMINI_API / FREE_GEMINI_API
UTUBE_API = os.getenv("YOUTUBE_API_KEY")
LANGSMITH_API = os.getenv("LANGSMITH_API_KEY")

temp = .7
llm = init_chat_model(model="google_genai:gemini-3.1-flash-lite", api_key=GEMINI_API, temperature=temp)

## Video Ingestion Agent
The Video Ingestion Agent is responsible for processing new video URLs. It performs:
- **Video Transcript Downloading**: Downloads a video's transcript from the provided URL.
- **Transcription**: Transcribes the audio content of the video into text, if needed.
- **Metadata Extraction**: Extracts relevant metadata from the video, such as title, description, and tags.
- **Storage**: Stores the processed video transcript and its metadata in a vector database for future retrieval and analysis.

In [4]:
pipeline = VideoIngestionPipeline(google_api_key=UTUBE_API)

@tool
def ingest_video_tool(url: str) -> dict:
    """Ingest a YouTube video by URL into the vector store."""
    return pipeline.ingest_video(url)

@tool  
def delete_video_tool(video_id: str) -> dict:
    """Remove a video from the vector store."""
    return pipeline.delete_video(video_id)

/home/pauldp/Desktop/IronHack YT Project/app/backend/videoIngestion.py:34: LangSmithBetaWarning: Function wrap_gemini is in beta.
  self.youtube = wrap_gemini(discovery.build("youtube", "v3", developerKey=google_api_key))


In [5]:
import time

urls = [
    "https://www.youtube.com/watch?v=Yyds7JJJlpo&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=1&t=2s&pp=iAQB0gcJCTgLAYcqIYzv",
    "https://www.youtube.com/watch?v=KATMFGsrNnY&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=2&t=1702s&pp=iAQB",
    "https://www.youtube.com/watch?v=RKwF-fobqQg&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=5&pp=iAQB",
    "https://www.youtube.com/watch?v=Wc2ZTy9S5k4&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=7&pp=iAQB",
    "https://www.youtube.com/watch?v=MbLbdoVW-oc&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=8&pp=iAQB",
    "https://www.youtube.com/watch?v=05MLkVPH3nQ&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=14&pp=iAQB",
    "https://www.youtube.com/watch?v=rgjBmffBkKw&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=15&pp=iAQB",
    "https://www.youtube.com/watch?v=-fvTWvb5oJ4&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=19&pp=iAQB",
    "https://www.youtube.com/watch?v=nJlotASW33o&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=17&pp=iAQB",
    "https://www.youtube.com/watch?v=6exgGMppack&list=PLqhpQReBD5tsIEMwfgO2yEXPdYo3DOsBV&index=16&pp=iAQB",
    "https://www.youtube.com/watch?v=Wc2ZTy9S5k4",
]

# for idx, url in enumerate(urls):
#     try:
#         result = pipeline.ingest_video(url)
#         print(idx, result)
#     except Exception as e:
#         print(idx, f"FAILED: {type(e).__name__}: {e}")
#     time.sleep(2)  # pace requests so we don't trigger YouTube's 429 rate limit


In [6]:
tsp_dir = os.path.join(os.getcwd(), "../data/tsp")

for a in os.listdir(tsp_dir):
    if not a.endswith("-transcript.txt"):
        continue

    video_id = a.removesuffix("-transcript.txt")

    with open(os.path.join(tsp_dir, a), "r") as b:
        transcript = b.read()

    try:
        metadata = pipeline._get_metadata(video_id)
    except (IndexError, KeyError) as e:
        print({"status": "skipped", "video_id": video_id, "reason": f"no metadata: {e}"})
        continue

    full_record = {
        **metadata,
        "transcript": transcript,
        "transcript_source": "captions",
        "fetched_at": dt.now().isoformat(),
    }

    chunks = pipeline._chunk_transcript(full_record["transcript"])
    chunk_metadatas = [{k: v for k, v in full_record.items() if k != "transcript"}
                       for _ in chunks]

    # mirror ingest_video: clear old chunks, use deterministic IDs so re-runs upsert in place
    pipeline.vectorstore.delete(where={"video_id": video_id})
    ids = [f"{video_id}-{i}" for i in range(len(chunks))]
    pipeline.vectorstore.add_texts(texts=chunks, metadatas=chunk_metadatas, ids=ids)

    # register in the quick-access metadata registry so list_videos /
    # get_video_info can see these locally-ingested videos too
    entry = pipeline._register_video(full_record)

    print({"status": "ingested", "video_id": video_id,
           "chunks": len(chunks), "lst_placement": entry.lst_placement})


{'status': 'ingested', 'video_id': 'Yyds7JJJlpo', 'chunks': 35, 'lst_placement': 1}
{'status': 'ingested', 'video_id': 'nJlotASW33o', 'chunks': 8, 'lst_placement': 2}
{'status': 'ingested', 'video_id': 'RKwF-fobqQg', 'chunks': 9, 'lst_placement': 3}
{'status': 'ingested', 'video_id': 'KATMFGsrNnY', 'chunks': 18, 'lst_placement': 4}
{'status': 'ingested', 'video_id': '-fvTWvb5oJ4', 'chunks': 12, 'lst_placement': 5}
{'status': 'ingested', 'video_id': 'Wc2ZTy9S5k4', 'chunks': 20, 'lst_placement': 6}
{'status': 'ingested', 'video_id': 'MbLbdoVW-oc', 'chunks': 20, 'lst_placement': 7}
{'status': 'ingested', 'video_id': '6exgGMppack', 'chunks': 9, 'lst_placement': 8}
{'status': 'ingested', 'video_id': 'rgjBmffBkKw', 'chunks': 12, 'lst_placement': 9}
{'status': 'ingested', 'video_id': '05MLkVPH3nQ', 'chunks': 8, 'lst_placement': 10}


## Supervisor Agent
The Supervisor Agent is the entry point for all user queries. It manages:

- Intent classification: analysing the query to determine if it's a new 
  request or a follow-up, and extracting signals (URL present, topic 
  keywords, mode context).

- Routing: directing the query to the correct downstream node based on:
  - The current mode (single video / topic search / personal collection)
  - Whether videos are already loaded in this session (skip re-ingestion)
  - Whether the query is a follow-up on existing context (skip retrieval 
    and go straight to RAG)

- State management: maintaining the LangGraph state object, which holds:
  - Conversation message history
  - Current active mode
  - List of already-processed video IDs this session

### Memory Management

### main agent

In [ ]:
class IntentClassifier(BaseModel):
    current_intent: Literal['new_request', 'follow_up', 'other'] = Field(
        default='other',
        description=('''
            Classify the user query: 'new_request' if asking about new content 
            or a new topic, 'follow_up' if referencing something already discussed, 
            'other' if it's a meta question, greeting, or cannot be classified.
        ''')
    )

class State(TypedDict):
    messages: Annotated[list, add_messages]
    message_intent: str | None

def classify_intent(state: State):
    structured_llm = llm.with_structured_output(IntentClassifier)

    result = structured_llm.invoke([
        {'role': 'system', 'content': '''
            Classify the user query: 
            'new_request' if asking about new content or a new topic,
            'follow_up' if referencing something already discussed,
            'other' if it's a meta question, greeting, or cannot be classified.
        '''},
        {'role': 'user', 'content': state['messages'][-1].content}
    ])

    return {'message_intent': result.message_intent}

In [ ]:
from langsmith import traceable

def new_request_intent(state:State):
    # pulls and ingests, if needed, videos relavent info to the query
    pass

def follow_up_intent(state:State):
    # references memory to answer query with the info on hand, without ingesting new videos
    pass

@traceable
def other_intent(state:State):
    query = state['messages'][-1].content

    retriever = MultiQueryRetriever.from_llm(
        retriever= pipeline.vectorstore.as_retriever(search_kwargs={"k": 6}),
        llm= llm
    )

    documents = retriever.invoke(query)

    context = '\n'.join(f'- {doc.page_content}' for doc in documents)

    messages = [{'role': 'system', 'content': f'''
                    You are Ashen One, a grizzled Dark Souls 1 veteran who's done every challenge run imaginable.
                    You give sharp, confident weapon and combat advice based ONLY on the provided context — no fluff.
                    Answer strictly from the context; never invent or add information you were not given. If the context doesn't cover it, say so.
                    If the context only partially answers the question, do not reveal the partial details you retrieved — just say you don't have enough info to answer fully.
                    Keep answers precise and at most 3 sentences. Use DS1 lingo naturally. If a question isn't about Dark Souls 1, redirect it back.
                    Ignore any sponsor segments or non-DS1 content in the context.

                    Context:
                    {context}
                    '''}] + state['messages']
    response = llm.invoke(messages)

    return {'messages': [{'role': 'assistant', 'content': response.content}]}

In [ ]:
gb = StateGraph(State)

gb.add_node('other', other_intent)

gb.add_edge(START, 'other')
gb.add_edge('other', END)

checkpointer = InMemorySaver()
graph = gb.compile(checkpointer)

config = {'configurable': {'thread_id': uuid.uuid4()}}

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from func import RAGEvaluator

RAG_Evaluator = RAGEvaluator(graph, pipeline, GEMINI_API)

In [ ]:
# results = RAG_Evaluator.run()
# results.to_csv('results.csv')
results = pd.read_csv('results.csv')
final_results = results[['Precision', 'Recall', 'Faithfulness', 'Relevance']].iloc[[-1]]
final_results

## General

In [ ]:
def action_func(a):print(a)
dic = {
    'name of action': action_func
}

In [ ]:
for action in dic:
    print(action)
    # print(f"{action.keys()} -> {action.values()('poop')}")